## craete text chunk for the dataset to create RAG
## tables info missing from samples.information_schema.columns see cell2
##alternative is ti query using SQL


In [0]:
%sql
select * from samples.information_schema.columns
where table_schema = 'tpcds_sf1000'


### Itterate through the list of tables to get columns names along with its dtype
### Convert to Panda dataframe to use iterows & reset_index

In [0]:
%sql
SHOW TABLES IN samples.tpcds_sf1000

In [0]:
table_names = [row.tableName for row in spark.sql("SHOW TABLES IN samples.tpcds_sf1000").collect()]
all_columns = []
for table in table_names:
    df = spark.table(f"samples.tpcds_sf1000.{table}")
    schema = df.schema
    for field in schema.fields:
        all_columns.append({
            "table_name": table,
            "column_name": field.name,
            "data_type": str(field.dataType)
        })
import pandas as pd

# Convert all_columns to a pandas DataFrame for easier text chunking
all_columns_df = pd.DataFrame(all_columns)

# Group columns by table and concatenate into schema chunks
rag_chunks = all_columns_df.groupby("table_name").apply(
    lambda x: ", ".join(f"{row['column_name']} {row['data_type']}" for _, row in x.iterrows())
).reset_index().rename(columns={0: "schema_chunk"})

display(rag_chunks)